# [데이터 통합] 이벤트 로그 & 교정 기록 병합

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

해결 하고자 하는 문제 = 재방문율
방향성을 가지고 데이터 탐색


In [ ]:
#필요없는 컬럼
drop_cols_df2 = [
    'browser_version', 'email', 'geo_source', 'name',
    'initial_utm_campaign', 'initial_utm_content', 'initial_utm_id',
    'initial_utm_medium', 'initial_utm_source', 'initial_utm_term',
    'initial_utm_source_platform'
]

drop_cols_df3 = [
    'browser_is_whale', 'browser_is_chrome', 'browser_is_edge',
    'email', 'name', 'browser_version', 'insert_id',
    'screen_height', 'screen_width', 'search_engine', 'lib_version',
    'mp_api_endpoint', 'mp_lib', 'mp_loader', 'extension_version',
    'llm_provider', 'llm_version', 'position', 'platform', 'trigger'
]

1번 데이터셋 불러오기  

In [ ]:
df1 = pd.read_json('../data/raw/1. 사용자별 사용량 데이터.json')

df1 데이터프레임의 '_id' 컬럼이 MongoDB 스타일의 ObjectID 형식   
JSON 파일을 pandas로 읽어들일 때, 이런 중첩된 딕셔너리 구조가 그대로 로드 하는것을 방지   
df1의 '_id' 컬럼을 문자열로 변환 후, '_id_str' 컬럼에 저장  
'recent_execution_date'날짜 값을 추출한 후 datetime 형식으로 변환  

In [ ]:
df1['_id_str'] = df1['_id'].apply(lambda x: x.get('$oid') if isinstance(x, dict) else None)
df1['recent_execution_date'] = df1['recent_execution_date'].apply(lambda x: x.get('$date') if isinstance(x, dict) else None)
df1['recent_execution_date'] = pd.to_datetime(df1['recent_execution_date'])

2번 데이터셋 불러오기  

In [ ]:
df2 = pd.read_json('../data/raw/2. 사용자 클라이언트 데이터.json', lines=True) 

In [ ]:
df2['user_id'].value_counts()

df2에 'properties' 컬럼을 중첩된 JSON으로 정규화  
'distinct_id'가 중복되면 삭제하고, 원본 df2와 병합해서 충돌 방지  
중첩된 속성을 평탄화  

In [ ]:
if 'properties' in df2.columns:
    properties_df = pd.json_normalize(df2['properties'])
    if 'distinct_id' in properties_df.columns:
        properties_df = properties_df.drop('distinct_id', axis=1)  
    df2 = pd.concat([df2.drop('properties', axis=1), properties_df], axis=1)

3번 데이터셋 불러오기

In [ ]:
df3 = pd.read_json('../data/raw/3. 이벤트 데이터.json', lines=True)

df3의 'user_id' 컬럼을 문자열로 변환 후 'user_id_str' 컬럼에 저장  
딕셔너리 형식에서 '$oid' 값을 추출

In [ ]:
df3['user_id_str'] = df3['user_id'].apply(lambda x: x.get('$oid') if isinstance(x, dict) else x)

df3에 'properties' 컬럼이 있으면 이를 정규화  
'distinct_id'와 'user_id'가 중복되면 삭제, 원본 df3와 병합  
df2와 유사한 평탄화  

In [ ]:
if 'properties' in df3.columns:
    props_df3 = pd.json_normalize(df3['properties'])
    for col in ['distinct_id', 'user_id']:
        if col in props_df3.columns:
            props_df3 = props_df3.drop(col, axis=1)
    df3 = pd.concat([df3.drop('properties', axis=1), props_df3], axis=1)

In [ ]:
df3

df1과 df2를 병합하여 user_merged 데이터프레임을 만듬  
'_id_str'과 'distinct_id'를 키로 사용하며, outer join으로 모든 레코드를 포함  
사용자 사용량과 클라이언트 데이터를 결합  

In [ ]:
user_merged = pd.merge(df1, df2, left_on='_id_str', right_on='distinct_id', how='outer')

In [ ]:
user_merged

df3과 user_merged에 병합 키('merge_key')를 생성(병합할때 안정성 확보)  
'user_id_str'을 사용  
left join으로 병합 후 full_merged를 만들고 불필요한 키 컬럼을 삭제  
이벤트 데이터와 사용자 데이터를 통합  

In [ ]:
df3['merge_key'] = df3['user_id_str'].fillna(df3['distinct_id'])
user_merged['merge_key'] = user_merged['_id_str'].fillna(user_merged['distinct_id'])
full_merged = pd.merge(df3, user_merged, on='merge_key', how='left', suffixes=('_event', '_user'))
full_merged.drop(['merge_key_event', 'merge_key_user', 'merge_key'], axis=1, errors='ignore', inplace=True)

In [ ]:
full_merged

중복된 컬럼 이름을 검사, 중복 시 숫자 접미사를 추가해서 고유화  
해시 가능한 컬럼를 선택, 처음 10개 컬럼으로 중복 행을 제거  
데이터 중복을 정리

In [ ]:
from collections import Counter
col_count = Counter(full_merged.columns)
if any(count > 1 for count in col_count.values()):
    new_cols = []
    seen = {}
    for col in full_merged.columns:
        if col in seen:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")
        else:
            seen[col] = 0
            new_cols.append(col)
    full_merged.columns = new_cols

hashable_cols = [col for col in full_merged.columns 
                 if full_merged.dtypes[col] in ['int64', 'float64', 'object', 'datetime64[ns]'] 
                 and not full_merged[col].apply(lambda x: isinstance(x, (dict, list))).any()]

full_merged = full_merged.drop_duplicates(subset=hashable_cols[:10])

데이터 클리닝  
'time' 컬럼을 한국 시간으로 변환  
NaN 값을 채우며(지리 정보는 'Unknown', 응답 시간은 중위수, 입력 길이는 0)  
불필요한 컬럼을 삭제  
익명 사용자 플래그('is_anon') 추가, 'count' 컬럼을 정수형으로 변환  
이상치 클리핑

In [ ]:
if 'time' in full_merged.columns:
    full_merged['time_dt'] = pd.to_datetime(full_merged['time'], unit='s', utc=True).dt.tz_convert('Asia/Seoul')

if 'recent_execution_date' in full_merged.columns:
    full_merged['recent_execution_date'] = full_merged['recent_execution_date'].apply(lambda x: x.get('$date') if isinstance(x, dict) else None)
    full_merged['recent_execution_date'] = pd.to_datetime(full_merged['recent_execution_date'])

if 'mp_api_timestamp_ms' in full_merged.columns:
    full_merged.drop('mp_api_timestamp_ms', axis=1, inplace=True)

geo_cols = ['city', 'region', 'country_code', 'timezone']
for col in geo_cols:
    if col in full_merged.columns:
        full_merged[col] = full_merged[col].fillna('Unknown')

if 'response_time_ms' in full_merged.columns:
    median_response = full_merged['response_time_ms'].median()
    full_merged['response_time_ms'] = full_merged['response_time_ms'].fillna(median_response)

if 'input_sentence_length' in full_merged.columns:
    full_merged['input_sentence_length'] = full_merged['input_sentence_length'].fillna(0)

if 'browser' in full_merged.columns:
    full_merged['browser'] = full_merged['browser'].fillna('Unknown')

drop_cols = ['mp_lib', 'mp_api_endpoint', 'mp_loader', 'lib_version', 'mp_processing_time_ms', 'insert_id', 'device_id_event', '_id', '_id_str']
full_merged = full_merged.drop(columns=[col for col in drop_cols if col in full_merged.columns], errors='ignore')

if 'distinct_id' in full_merged.columns:
    full_merged['is_anon'] = full_merged['distinct_id'].str.startswith('$device:')

if 'count' in full_merged.columns:
    full_merged['count'] = full_merged['count'].astype('Int64')

if 'response_time_seconds' in full_merged.columns:
    full_merged['response_time_seconds'] = pd.to_numeric(full_merged['response_time_seconds'], errors='coerce')
    
if 'count' in full_merged.columns:
    percentile_99 = full_merged['count'].quantile(0.99)
    full_merged['count'] = full_merged['count'].clip(upper=percentile_99)

print("Shape after cleaning:", full_merged.shape)
print(full_merged.info())
full_merged.head()

다 같이 필요 없다고 정의한 drop_cols_df2와 drop_cols_df3를 합쳐 중복 제거한 리스트를 만들고, full_merged에서 해당 컬럼들을 삭제

In [ ]:
drop_cols = list(set(drop_cols_df2 + drop_cols_df3))

full_merged = full_merged.drop(columns=[col for col in drop_cols if col in full_merged.columns])

In [ ]:
full_merged.info()

병합된 데이터프레임을 'full_merged.csv' 파일로 저장

In [ ]:
full_merged.to_csv('full_merged.csv', index=False, encoding='utf-8-sig')

In [ ]:
print("Usage Statistics:")
print(user_merged['count'].describe())